<a href="https://colab.research.google.com/github/khushboopandey6/CS682-Synthetic-Data-Project/blob/main/01_data_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers datasets accelerate
!pip install scikit-learn pandas numpy
!pip install sentence-transformers evaluate

In [2]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import torch

PROJECT_PATH = "/content/drive/MyDrive/CS682_Neural_Project"

os.makedirs(PROJECT_PATH, exist_ok=True)

print("GPU Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

GPU Available: True
Tesla T4


In [5]:
from datasets import load_dataset

dataset = load_dataset("PolyAI/banking77")

README.md:   0%|          | 0.00/13.0k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  294kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 92.4kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/10003 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3080 [00:00<?, ? examples/s]

In [6]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 10003
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 3080
    })
})


In [7]:
print(dataset["train"][0])

{'text': 'I am still waiting on my card?', 'label': 11}


In [8]:
# Get the names of all intent classes
class_names = dataset["train"].features["label"].names

print("Total classes:", len(class_names))

print("\nFirst 10 classes:")
print(class_names[:10])

Total classes: 77

First 10 classes:
['activate_my_card', 'age_limit', 'apple_pay_or_google_pay', 'atm_support', 'automatic_top_up', 'balance_not_updated_after_bank_transfer', 'balance_not_updated_after_cheque_or_cash_deposit', 'beneficiary_not_allowed', 'cancel_transfer', 'card_about_to_expire']


In [10]:
import os

os.makedirs(
    "/content/drive/MyDrive/CS682_Neural_Project/data/raw",
    exist_ok=True
)

print("Folder created successfully!")

Folder created successfully!


In [11]:
import json
import os

class_names = dataset["train"].features["label"].names

metadata = {
    "total_classes": len(class_names),
    "train_examples": len(dataset["train"]),
    "test_examples": len(dataset["test"]),
    "class_names": class_names
}

metadata_path = os.path.join(
    PROJECT_PATH,
    "data",
    "raw",
    "dataset_metadata.json"
)

with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=4)

print("Dataset metadata saved successfully!")
print(metadata_path)

Dataset metadata saved successfully!
/content/drive/MyDrive/CS682_Neural_Project/data/raw/dataset_metadata.json


In [12]:
from collections import Counter

train_labels = dataset["train"]["label"]

class_counts = Counter(train_labels)

print("Number of classes:", len(class_counts))
print("Minimum examples in a class:", min(class_counts.values()))
print("Maximum examples in a class:", max(class_counts.values()))

Number of classes: 77
Minimum examples in a class: 35
Maximum examples in a class: 187


In [13]:
from collections import defaultdict
import random

# Set a random seed so we get the same selection every time
random.seed(42)

# Get the original training dataset
train_data = dataset["train"]

# These lists will store the selected validation examples
# and the examples left for training
validation_indices = []
remaining_indices = []

# Group all training examples based on their class labels
class_to_indices = defaultdict(list)

for index, example in enumerate(train_data):
    label = example["label"]
    class_to_indices[label].append(index)

# Select 10 random examples from each of the 77 classes
for label, indices in class_to_indices.items():

    # Randomly pick 10 examples for validation
    selected = random.sample(indices, 10)

    # Save their indices in the validation list
    validation_indices.extend(selected)

    # Put all other examples into the remaining training list
    for index in indices:
        if index not in selected:
            remaining_indices.append(index)

# Create the two datasets using the selected indices
validation_data = train_data.select(validation_indices)
remaining_train_data = train_data.select(remaining_indices)

# Print the results to check our split
print("Validation examples:", len(validation_data))
print("Remaining training examples:", len(remaining_train_data))
print("Number of validation classes:", len(set(validation_data["label"])))

Validation examples: 770
Remaining training examples: 9233
Number of validation classes: 77


In [14]:
from collections import Counter

# Count how many validation examples belong to each class
validation_counts = Counter(validation_data["label"])

# Check whether every class has exactly 10 examples
print("Are all classes having 10 examples?",
      all(count == 10 for count in validation_counts.values()))

print("Validation examples per class:", set(validation_counts.values()))

Are all classes having 10 examples? True
Validation examples per class: {10}


In [15]:
import random
from collections import defaultdict

# Set the random seed for reproducibility
random.seed(42)

# Group the remaining training examples by class
class_to_indices = defaultdict(list)

for index, example in enumerate(remaining_train_data):
    label = example["label"]
    class_to_indices[label].append(index)

# Store the selected examples for low-resource training
low_resource_indices = []

# Select 10 examples from each class
for label, indices in class_to_indices.items():
    selected = random.sample(indices, 10)
    low_resource_indices.extend(selected)

# Create the low-resource training dataset
low_resource_data = remaining_train_data.select(low_resource_indices)

# Check the dataset size
print("Low-resource training examples:", len(low_resource_data))
print("Number of classes:", len(set(low_resource_data["label"])))

Low-resource training examples: 770
Number of classes: 77


In [16]:
from collections import Counter

# Count examples in each class in the original training data
class_counts = Counter(dataset["train"]["label"])

# Sort classes from fewest examples to most examples
sorted_classes = sorted(class_counts.items(), key=lambda x: x[1])

# Select the 15 classes with the fewest examples
rare_classes = [label for label, count in sorted_classes[:15]]

print("Number of rare classes:", len(rare_classes))
print("\nRare class names:")

for label in rare_classes:
    print(class_names[label], "-", class_counts[label], "examples")

Number of rare classes: 15

Rare class names:
contactless_not_working - 35 examples
virtual_card_not_working - 41 examples
card_acceptance - 59 examples
card_swallowed - 61 examples
lost_or_stolen_card - 82 examples
compromised_card - 86 examples
atm_support - 87 examples
receiving_money - 95 examples
top_up_limits - 97 examples
get_disposable_virtual_card - 97 examples
getting_virtual_card - 98 examples
unable_to_verify_identity - 102 examples
topping_up_by_card - 103 examples
verify_my_identity - 104 examples
passcode_forgotten - 105 examples


In [17]:
import random
from collections import defaultdict

# Set a random seed so the selection is reproducible
random.seed(42)

# Group the remaining training examples by class
class_to_indices = defaultdict(list)

for index, example in enumerate(remaining_train_data):
    label = example["label"]
    class_to_indices[label].append(index)

# Store the selected examples
imbalanced_indices = []

# Select fewer examples for rare classes
# Select more examples for the other classes
for label, indices in class_to_indices.items():

    if label in rare_classes:
        number_to_select = 5
    else:
        number_to_select = 20

    selected = random.sample(indices, number_to_select)
    imbalanced_indices.extend(selected)

# Create the imbalanced training dataset
imbalanced_data = remaining_train_data.select(imbalanced_indices)

# Display the results
print("Imbalanced training examples:", len(imbalanced_data))
print("Number of classes:", len(set(imbalanced_data["label"])))

Imbalanced training examples: 1315
Number of classes: 77


In [18]:
# Save the prepared datasets to Google Drive

validation_data.save_to_disk(
    PROJECT_PATH + "/data/processed/validation_data"
)

low_resource_data.save_to_disk(
    PROJECT_PATH + "/data/processed/low_resource_data"
)

imbalanced_data.save_to_disk(
    PROJECT_PATH + "/data/processed/imbalanced_data"
)

remaining_train_data.save_to_disk(
    PROJECT_PATH + "/data/processed/remaining_train_data"
)

print("All datasets saved successfully!")

Saving the dataset (0/1 shards):   0%|          | 0/770 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/770 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1315 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/9233 [00:00<?, ? examples/s]

All datasets saved successfully!
